# 04 — Non-triviality on real groups (RQ2)
Tests whether GUARD ($\kappa$, $C\cdot\kappa$) separates structurally-contaminated groups from diffusely-contaminated ones **beyond** individual-score aggregation (mean/max/p90). Structure is measured from clean labels: a group's contamination is 'structured' when its misregistered items concentrate on one true category, 'diffuse' when they scatter.

In [1]:
# Notebook: 04_nontriviality_rq2
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

import sys; sys.path.append(os.path.join("..", "src"))
import numpy as np
from guard_core import axis_A, axis_B, individual_scores
from sklearn.metrics import roc_auc_score
DATA_DIR = os.path.join("..", "data")
meta = pd.read_parquet(os.path.join(DATA_DIR, "item_meta.parquet"))
pi = np.load(os.path.join(DATA_DIR, "pi_memmap.npy"), mmap_mode="r")
N, K = pi.shape
group_ids = meta["noisy_id"].values; a = meta["noisy_id"].values
valid = meta["clean_id"].values >= 0

# --- label each group's contamination structure from clean labels ---
def structure_of_group(idx):
    """Return (n_mis, structure_score). structure_score in [0,1]:
    fraction of misregistered items falling in their single most common TRUE category."""
    ci = meta["clean_id"].values[idx]; ni = meta["noisy_id"].values[idx]
    mis = idx[(ci >= 0) & (ci != ni)]
    if len(mis) < 5:
        return len(mis), np.nan
    true_of_mis = meta["clean_id"].values[mis]
    top = np.bincount(true_of_mis).max()
    return len(mis), top / len(mis)

sizes = pd.Series(group_ids).value_counts()
keep = sizes[(sizes >= 30) & (sizes <= 5000)].index
recs = []
for gid in keep:
    idx = np.where(group_ids == gid)[0]
    n_mis, struct = structure_of_group(idx)
    if np.isnan(struct):
        continue
    Pg = np.asarray(pi[idx], dtype=np.float64)
    W, D, _, _ = axis_A(Pg); C, kappa, _, _ = axis_B(Pg, a[idx], K)
    s = individual_scores(Pg, a[idx])
    recs.append(dict(group=gid, n=len(idx), n_mis=n_mis, structure=struct,
                     mean_s=s.mean(), max_s=s.max(), p90_s=np.quantile(s, 0.9),
                     C=C, kappa=kappa, C_kappa=C * kappa))
R = pd.DataFrame(recs)
R.to_csv(os.path.join(TAB, "t_rq2_real_groups.csv"), index=False)
print(f"groups analyzed: {len(R):,}")
print(R[["structure", "kappa", "C", "mean_s"]].describe().to_string())

# --- structured (top tercile) vs diffuse (bottom tercile) separability ---
lo, hi = R["structure"].quantile(1/3), R["structure"].quantile(2/3)
sub = R[(R["structure"] <= lo) | (R["structure"] >= hi)].copy()
y = (sub["structure"] >= hi).astype(int).values
metrics = ["mean_s", "max_s", "p90_s", "C", "kappa", "C_kappa"]
def sep(m):
    au = roc_auc_score(y, sub[m].values); return round(max(au, 1 - au), 3)
tab = pd.DataFrame({"metric": metrics, "separability": [sep(m) for m in metrics]})
tab.to_csv(os.path.join(TAB, "t_rq2_real_auroc.csv"), index=False)
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(5.6, 3.4))
colors = [GREYS[3]] * 4 + [GREYS[0]] * 2
ax.bar(range(len(metrics)), tab["separability"], color=colors, edgecolor="black", linewidth=0.6)
ax.axhline(0.5, color="black", ls=":", lw=1)
ax.set_xticks(range(len(metrics)))
ax.set_xticklabels(["mean","max","p90","C","kappa","C*kappa"], rotation=30, ha="right")
ax.set_ylabel("separability (max(AUROC,1-AUROC))"); ax.set_ylim(0.4, 1.02)
savefig(fig, "f_rq2_real_separation")
print("saved f_rq2_real_separation.{png,pdf}")

groups analyzed: 1,159
         structure        kappa            C       mean_s
count  1159.000000  1159.000000  1159.000000  1159.000000
mean      0.638665     0.384531     0.561892     0.698384
std       0.266254     0.144334     0.286404     0.266910
min       0.086957     0.106994     0.019487     0.038435
25%       0.400000     0.270670     0.306036     0.493361
50%       0.636364     0.369533     0.579415     0.775695
75%       0.891598     0.479978     0.829737     0.938088
max       1.000000     0.827387     0.999027     0.999864
 metric  separability
 mean_s         0.695
  max_s         0.640
  p90_s         0.703
      C         0.695
  kappa         0.634
C_kappa         0.592
saved f_rq2_real_separation.{png,pdf}
